# Steps for the first run

- Run `conda install -c conda-forge r-plyr r-dplyr r-data.table r-bit64 r-geosphere r-tidyverse r-maps r-scales r-stringr r-ggplot2` using command line and restart the R kernel of this notebook
- Run the next `install.packages(c("feather", "geosphere")) cell` - required for some reason
- For some reason the notebook does not work beyond cell 16. Use `nbconvert` in command line, or nbconvertR package in R to convert this notebook to a .r file. Either use R prompt or use `Rscript` command to run the generated .r file. To successfully execute nbconvert in R:
    - Install package nbconvertR using `conda install -c r r-nbconvertr`
    - `library(nbconvertR)`
    - `nbconvert("postprocess_new_adsb.ipynb", "script")`
    - This will produce postprocess_new_adsb.r that can either be sourced from R prompt using `source` or from command line using `Rscript`

In [ ]:
# install.packages(c("feather", "geosphere"))

In [ ]:
# library(nbconvertR)
# nbconvert("01-label_aircraft_id.ipynb", "script")

# Data dictionary

| Variable | Description |
|----------|-------------|
|    V1    | Unknown timestamp |
|    V2    | Seems to change for each aircraft during exit-entry |
|    V3    | Looks like a proxy for aircraft |
|    V4    | |
|    V5    | |
|    V6    | Heading direction in degrees, ranges from 1 to 360 |
|    V7    | (Barometric?) Altitude in ft |
|    V8    | (Estimated?) Ground speed in kts? |
|    V9    | |
|    V10   | Call-sign: not a good proxy for aircraft as one might expect |
|    V11   | |
|    V12   | |
|    V13   | Useful timestamp? |
|    V14   | Source airport code |
|    V15   | Destination airport code |
|    V16   | Another proxy for aircraft? |
|    V17   | Flag indicating if the aircraft is at an airport? |
|    V18   | Instantaneous climb rate (ft/s)? (jittery) |
|    V19   | Another proxy for aircraft? |
|  V16_V19 | Looks like a good proxy for aircraft ID. Can be used to refine V3 further |

# Definitions

| Term | Definition |
|------|------------|
| Maximum time between two samples for the aircraft to be declared "expired" | 2 hours |
| Time from exit to re-entry as a new aircraft | 18 mins |
| Minimum time to remain on ground to be labeled as a new aircraft | 18 mins |
| Absolute latitude difference around airport for an aircraft to be labeled 'landed' | 0.1&deg; |
| Absolute longitude difference around airport for an aircraft to be labeled 'landed' | 0.1&deg; |

# Initialization

In [ ]:
lower_lat <- 35
upper_lat <- 45
lower_lon <- -78
upper_lon <- -68
lat_range <- 0.4
lon_range <- 0.5
ts_diff_ground <- 1080
ts_diff_exit_enter <- 1080
ts_diff_expired <- 7200
airport_lat_half_range <- airport_lon_half_range <- 0.1

# Constraints

In [ ]:
ground_speed_ground_max_kts <- 50 # Ground-speed limit used with zero-altitude evidence
landed_altitude <- 1000 # may not need this because of V17
surely_grounded_speed <- 10 # Low-speed hint; not sufficient alone to label ground
altitude_ceiling <- 48000

# Loading packages and utilities

In [ ]:
suppressMessages({
    library(plyr)
    library(dplyr)
    library(data.table)
    library(bit64)
    library(geosphere)
    library(feather)
    # library(rnaturalearth)
})

source("preprocess_utils.R")

In [ ]:
# ==============================================================================
# process_ac_data(): rename, clean, and de-duplicate one ADS-B CSV chunk
# ==============================================================================
process_ac_data <- function(ac_data) {
  ac_data$alt_baro_raw <- as.character(ac_data$alt_baro)
  ac_data$ground_flag_raw <- !is.na(ac_data$alt_baro_raw) &
    tolower(trimws(ac_data$alt_baro_raw)) == "ground"
  ac_data <- ac_data %>% rename(

    # azimuth      = V6,  source       = V14, destination  = V15,
    # altitude     = V7,  ground_speed = V8,  aircraft     = V3,
    # ts           = V13, climb_rate   = V18, ground_flag  = V17,
    # lat          = V4,  lon          = V5,  aircraft_type = V11,
    altitude     = alt_baro, azimuth = track, ground_speed = gs,
    aircraft     = hex
  )
  ac_data$altitude_raw_ft <- suppressWarnings(
    as.numeric(as.character(ac_data$altitude))
  )
  ac_data$altitude <- ac_data$altitude_raw_ft
  ac_data$ground_speed <- suppressWarnings(
    as.numeric(as.character(ac_data$ground_speed))
  )

  # ── Conservative ground-state evidence ───────────────────────────────────────
  condition <- is.na(ac_data$azimuth) & !is.na(ac_data$true_heading)
  ac_data$azimuth[condition] <- ac_data$true_heading[condition]
  explicit_ground <- !is.na(ac_data$ground_flag_raw) & ac_data$ground_flag_raw
  zero_altitude_slow <- is.finite(ac_data$altitude_raw_ft) &
    ac_data$altitude_raw_ft == 0 & is.finite(ac_data$ground_speed) &
    ac_data$ground_speed <= ground_speed_ground_max_kts
  low_speed_only <- is.finite(ac_data$ground_speed) &
    ac_data$ground_speed <= surely_grounded_speed & !zero_altitude_slow &
    !explicit_ground
  ac_data$ground_flag <- explicit_ground | zero_altitude_slow
  ac_data$ground_flag_confidence <- ifelse(
    explicit_ground, 1.0,
    ifelse(zero_altitude_slow, 0.8, ifelse(low_speed_only, 0.25, 0.0))
  )
  ac_data$ground_state_uncertain <- low_speed_only
  ac_data$altitude[ac_data$ground_flag] <- 0


  # Remove rows flagged as estimated / interpolated positions
  # ac_data <- ac_data[ac_data$V10 != "F-EST", ]

  # Prefer the most complete record when a timestamp is duplicated.
  ac_data[, .dedup_score :=
    as.integer(is.finite(as.numeric(as.character(lon))) &
               is.finite(as.numeric(as.character(lat)))) +
    as.integer(is.finite(altitude_raw_ft)) +
    as.integer(is.finite(ground_speed)) +
    as.integer(is.finite(as.numeric(as.character(azimuth))))]
  setorder(ac_data, aircraft, ts, -.dedup_score)
  ac_data <- ac_data[, .SD[1], by = list(aircraft, ts)]
  ac_data[, .dedup_score := NULL]


  # Human-readable timestamp
  ac_data$ts_readable <- as.POSIXct(as.numeric(ac_data$ts)/1000, origin = "1970-01-01")

  # Heading quadrant 1–4
  ac_data$quadrant <- ceiling(ac_data$azimuth / 90)

  # Remove FAA test / ghost transmissions
  ac_data <- ac_data[!grepl("^FAA", ac_data$aircraft), ]


  return(ac_data)
}

In [ ]:
# ==============================================================================
# Load and merge all ADS-B CSV files
# ==============================================================================
data_dir <- "../data"  # downloader outputs; avoids copying large source CSVs
csv_files <- list.files(path = data_dir, pattern = "^ads-b.*\\.csv$", full.names = TRUE)
message("Found ", length(csv_files), " file(s): ", paste(csv_files, collapse = ", "))

dt_lst <- vector("list", length(csv_files))
for (i in seq_along(csv_files)) {
  dt_lst[[i]] <- process_ac_data(fread(csv_files[i], header = TRUE))
}

In [ ]:
ac_data <- rbindlist(dt_lst, fill = TRUE)
rm(dt_lst)
gc(verbose = FALSE)

message("Rows after rbindlist:               ", format(nrow(ac_data), big.mark=","),
        "  |  unique aircraft: ", format(uniqueN(ac_data$aircraft), big.mark=","))

setorder(ac_data, aircraft, ts)

# Add diagnostics without dropping or interpolating source samples.
ac_data <- annotate_track_quality(ac_data)
message("Rows with one or more QC flags: ",
        format(sum(ac_data$quality_flag_count > 0, na.rm = TRUE), big.mark=","))
message("Track segments after long-gap annotation: ",
        format(uniqueN(ac_data$track_segment_id), big.mark=","))


In [ ]:
head(ac_data)

In [ ]:
# ==============================================================================
# Per-aircraft lag/lead feature engineering
# ==============================================================================
# For each row, compute the previous row's values (last_*) and the next row's
# values (next_*) within the same aircraft.
#
# Lag convention: first row uses its own value (so last_ts_diff[1] = 0, not NA).
# Lead convention: last row gets NA (no subsequent observation).
#
# Note: source and destination are too sparse (<= 1 %) in the current dataset
# to be reliable session-boundary signals; they have been removed from the
# lag/lead columns and from the per-aircraft summaries.

ac_data[,
  c(
    # ── lag columns ────────────────────────────────────────────────────────────
    "last_lat", "last_lon", "last_azimuth", "last_altitude",
    "last_ts",# "last_V9",
    "last_ground_flag", "last_quadrant", "last_ground_speed",# "last_V2",
    # ── lead columns ───────────────────────────────────────────────────────────
    "next_lat", "next_lon", "next_azimuth", "next_altitude",
    "next_ts",# "next_V9",
    "next_ground_flag", "next_quadrant", "next_ground_speed",# "next_V2",
    # ── per-aircraft summaries ─────────────────────────────────────────────────
    "max_ac_alt"
  ) :=
  list(
    # lag
    c(lat[1],          lat[-.N]),          c(lon[1],          lon[-.N]),
    c(azimuth[1],      azimuth[-.N]),      c(altitude[1],     altitude[-.N]),
    c(ts[1],           ts[-.N]),
    #c(NA_real_,        V9[-.N]),
    c(ground_flag[1],  ground_flag[-.N]),
    c(quadrant[1],     quadrant[-.N]),
    c(ground_speed[1], ground_speed[-.N]),
    #c(V2[1],           V2[-.N]),

    # lead — B1 FIX: was `lat[-.1]` and `azimuth[-.1]` (decimal zero → all NA)
    c(lat[-1],         NA_real_),          c(lon[-1],         NA_real_),
    c(azimuth[-1],     NA_real_),          c(altitude[-1],    NA_real_),
    c(ts[-1],          NA_real_),
    #c(V9[-1],          NA_real_),
    c(ground_flag[-1], NA),
    c(quadrant[-1],    NA_integer_),
    c(ground_speed[-1], NA_real_),
    #c(V2[-1],          NA_character_),

    # summaries
    suppressWarnings(max(altitude_raw_ft, na.rm = TRUE))

  ),
  by = "aircraft"
]

# Keep ground-only tracks in the canonical dataset; flag them for downstream selection.
ac_data[!is.finite(max_ac_alt), max_ac_alt := NA_real_]
ac_data[, aircraft_ever_airborne := is.finite(max_ac_alt) & max_ac_alt > 0]
message("Ground-only aircraft retained: ",
        format(uniqueN(ac_data$aircraft[!ac_data$aircraft_ever_airborne]), big.mark=","))


# Time between consecutive samples
ac_data$last_ts_diff <- ac_data$ts - ac_data$last_ts
ac_data$next_ts_diff <- ac_data$next_ts - ac_data$ts

# ── Time-aware vertical rate (ft/min) ─────────────────────────────────────────
# Long gaps and duplicate timestamps remain missing rather than being treated
# as a measured climb or descent.
# `altitude_rate_fpm` is computed in annotate_track_quality() with ts converted
# from milliseconds to seconds. Do not divide by an ambiguous source climb-rate
# field: input schemas use different names and may use different units.
ac_data$est_climb_rate <- ac_data$altitude_rate_fpm


# Change in heading quadrant
ac_data$quadrant_change <- (ac_data$quadrant != ac_data$last_quadrant)

# Reading airports data and checking

In [ ]:
# ==============================================================================
# Airport database
# ==============================================================================
airports <- read.csv("../data/GlobalAirportDatabase.txt", header = FALSE,
                     stringsAsFactors = FALSE, sep = ":")
airports <- airports %>% dplyr::rename(airport = V2, lat = V15, lon = V16)
airports$airport <- toupper(airports$airport)
airports <- airports[(airports$airport != "") & (airports$airport != "N/A"), ]
airports <- airports[
  data.table::between(airports$lat, lower_lat, upper_lat) &
  data.table::between(airports$lon, lower_lon, upper_lon), ]
rownames(airports) <- NULL
message("Regional airports loaded: ", nrow(airports))

In [ ]:
table(ac_data$altitude[ac_data$ground_flag == 1])

In [ ]:
sum(ac_data$ground_flag, na.rm = T)

In [ ]:
sum((ac_data$last_ground_flag == 0) & (ac_data$ground_flag == 1), na.rm = T)

# Defining landing and entry into sector

In [ ]:
# ==============================================================================
# Touchdown detection
# ==============================================================================

# B7 FIX: mark the actual transition row (last_ground_flag = 0 → ground_flag = 1).
# The original code shifted the flag one row forward via c(F, touchdown[-.N]),
# placing it on the second grounded observation instead.  Aircraft whose last
# recorded row was the transition row silently lost their touchdown.
ac_data$touchdown_candidate <- (ac_data$last_ground_flag == 0) &
                                (ac_data$ground_flag == 1)
ac_data$touchdown <- ac_data$touchdown_candidate


In [ ]:
# Validate each touchdown against the altitude reached since the prior one.
ac_data <- annotate_touchdown_segments(ac_data)
# Discard touchdowns where the segment never reached 10 % of the aircraft's
# overall maximum altitude (e.g. spurious ground_flag flips at low altitude)
low_seg <- ac_data$touchdown &
           is.finite(ac_data$max_alt_touchdown_seg) &
           is.finite(ac_data$max_ac_alt) &
           (ac_data$max_alt_touchdown_seg < 0.1 * ac_data$max_ac_alt)
ac_data$touchdown[low_seg] <- FALSE

Consecutive landings within 10 mins is not possible. These can be considered as missed approaches, therefore only the final landing is considered

In [ ]:
# Recompute segment summaries after pruning touchdown candidates.
ac_data <- annotate_touchdown_segments(ac_data)

Something's wrong with the R kernel in a notebook. So, the following lines were executed in R shell.

In [ ]:
# Iteratively remove consecutive touchdowns < 10 min apart (missed approaches).
# The repeat loop converges because each pass can only reduce the row count.
touchdown_df <- ac_data[ac_data$touchdown, .(aircraft, ts)]
repeat {
  n_before <- nrow(touchdown_df)
  touchdown_df[,
    next_touchdown_ts := c(ts[-1], NA_real_),
    by = "aircraft"
  ]
  touchdown_df[,
    gap_ms := fifelse(is.na(next_touchdown_ts), 600000,
                      as.numeric(next_touchdown_ts) - as.numeric(ts))
  ]
  touchdown_df <- touchdown_df[gap_ms >= 600000, ]

  if (nrow(touchdown_df) == n_before) break
}

In [ ]:
# Update refined touchdown flags back directly in place
ac_data[, touchdown := FALSE]
ac_data[touchdown_df, touchdown := TRUE, on = .(aircraft, ts)]
message("Touchdowns after refinement:       ", sum(ac_data$touchdown))
message("Rows after touchdown update:        ", format(nrow(ac_data), big.mark=","))

invisible(gc(full = TRUE))


In [ ]:
touchdown_df

# Testing definition of touchdown

In [ ]:
# ==============================================================================
# Touchdown airport matching — vectorised Haversine via geosphere::distm
# ==============================================================================
# Replaces the slow row-by-row lapply/identify_airport() approach.
# distm(from, to) → n_touchdowns × n_airports distance matrix (metres).
# max.col(-dist_mat) finds the nearest airport for every touchdown at once.
# ap_coords is kept in scope and reused by the takeoff matching cell below.

ap_coords <- cbind(airports$lon, airports$lat)

td_idx <- which(ac_data$touchdown)
message("Touchdown candidates before airport filter: ", length(td_idx))

if (length(td_idx) > 0) {
  td_coords    <- cbind(ac_data$lon[td_idx], ac_data$lat[td_idx])
  # ── Chunked distm to avoid OOM on large datasets ─────────────────────────
  CHUNK_SIZE   <- 5000L
  n_td         <- length(td_idx)
  nearest_idx  <- integer(n_td)
  nearest_dist <- numeric(n_td)
  for (chunk_start in seq(1L, n_td, by = CHUNK_SIZE)) {
    chunk_end  <- min(chunk_start + CHUNK_SIZE - 1L, n_td)
    chunk_rows <- chunk_start:chunk_end
    d <- distm(td_coords[chunk_rows, , drop = FALSE], ap_coords)
    nearest_idx[chunk_rows]  <- max.col(-d)
    nearest_dist[chunk_rows] <- d[cbind(seq_len(nrow(d)), nearest_idx[chunk_rows])]
    rm(d)
  }
  # ─────────────────────────────────────────────────────────────────────────
  ac_data$touchdown_airport_distance_m <- NA_real_
  ac_data$touchdown_airport_distance_m[td_idx] <- nearest_dist

  within_10km  <- nearest_dist <= 10000

  message(sprintf("Touchdowns within 10 km of a known airport: %.1f %%",
                  100 * mean(within_10km)))

  # Remove touchdowns that don't resolve to any regional airport
  ac_data$touchdown[td_idx[!within_10km]] <- FALSE

  # Assign airport codes via pre-built character vector + := to avoid list column
  airport_vec <- character(nrow(ac_data))
  airport_vec[td_idx[within_10km]] <- airports$airport[nearest_idx[within_10km]]
  ac_data[, airport := airport_vec]
  rm(airport_vec)
} else {
  message("No touchdown candidates found")
  ac_data[, airport := character(nrow(ac_data))]
}
message("Touchdown rows with airport code assigned: ", sum(nchar(ac_data$airport) > 0))
ac_data$touchdown_event_confidence <- ifelse(
  ac_data$touchdown,
  pmax(ac_data$ground_flag_confidence, 0.8),
  ifelse(ac_data$touchdown_candidate, 0.25, 0.0)
)


In [ ]:
# ==============================================================================
# Sector entry and signal-expiry flags
# ==============================================================================
 
# B2 + B3 FIX:
#   B2 — the original first condition wrapped `|` inside an extra parenthesis,
#        making it evaluate as `ln < (numeric | logical)` ≈ `ln < 1` = always TRUE.
#   B3 — conditions 3 and 4 used `last_ln` (longitude) instead of `last_lt` (latitude).
lat_lon_boundary_flag <- function(lt, ln, last_lt, last_ln,
                                  lower_lt = lower_lat, lower_ln = lower_lon,
                                  upper_lt = upper_lat, upper_ln = upper_lon,
                                  lt_range = lat_range, ln_range = lon_range) {
  near_west  <- (ln      < lower_ln + ln_range) | (last_ln < lower_ln + ln_range)
  near_east  <- (ln      > upper_ln - ln_range) | (last_ln > upper_ln - ln_range)
  near_south <- (lt      < lower_lt + lt_range) | (last_lt < lower_lt + lt_range)  # B3
  near_north <- (lt      > upper_lt - lt_range) | (last_lt > upper_lt - lt_range)  # B3
  return(near_west | near_east | near_south | near_north)
}
 
ac_data$landed <- ac_data$touchdown
 
# "Entered": near-boundary position AND a long gap since the last sample
boundary_flag     <- lat_lon_boundary_flag(
  ac_data$lat, ac_data$lon, ac_data$last_lat, ac_data$last_lon
)
# ts is epoch milliseconds, so convert the thresholds from seconds.
ac_data$entered   <- !is.na(boundary_flag) & boundary_flag &
                     !is.na(ac_data$last_ts_diff) &
                     (ac_data$last_ts_diff > ts_diff_exit_enter * 1000)
 
# "Expired": no signal for more than 2 hours
ac_data$expired   <- !is.na(ac_data$last_ts_diff) &
                     (ac_data$last_ts_diff > ts_diff_expired * 1000)
 
 
# ==============================================================================
# Assign unique per-flight-leg ID
# ==============================================================================
# A new flight leg starts at each: landing, sector entry, or signal expiry.
# num_id is a per-aircraft counter; the composite id is globally unique.
 
ac_data$new_leg <- ac_data$landed | ac_data$entered | ac_data$expired
ac_data[, num_id := cumsum(new_leg), by = "aircraft"]
ac_data$id <- paste0(ac_data$aircraft, "_", ac_data$num_id)
message("Rows after session assignment:      ", format(nrow(ac_data), big.mark=","))
message("Unique flight sessions (id):        ", format(uniqueN(ac_data$id), big.mark=","))

In [ ]:
# ==============================================================================
# Takeoff detection and session-level airport propagation
# ==============================================================================
# Takeoff mirrors touchdown: ground_flag 1→0 (on-ground → airborne) with a
# positive altitude confirms the aircraft has left the runway.
# Uses the same numeric ground_flag convention as the touchdown cell (22):
#   ground_flag == 1  →  on ground
#   ground_flag == 0  →  airborne
#
# After matching takeoff locations to the nearest regional airport, both
# dest_airport (from touchdown) and src_airport (from takeoff) are broadcast to
# EVERY row in the flight session (by = id) so models see the full context.

# ── Takeoff flag ───────────────────────────────────────────────────────────────
ac_data$takeoff_candidate <- !is.na(ac_data$last_ground_flag) &
                             !is.na(ac_data$ground_flag) &
                             (ac_data$last_ground_flag == 1) &
                             (ac_data$ground_flag == 0) &
                             is.finite(ac_data$altitude_raw_ft) &
                             (ac_data$altitude_raw_ft > 0)
ac_data$takeoff <- ac_data$takeoff_candidate
ac_data[, previous_ground_confidence :=
          data.table::shift(ground_flag_confidence), by = aircraft]
ac_data$takeoff_event_confidence <- ifelse(
  ac_data$takeoff_candidate,
  pmax(ac_data$previous_ground_confidence, 0.8),
  0.0
)
message("Takeoff candidates detected:         ",
        sum(ac_data$takeoff_candidate, na.rm = TRUE))


# ── Vectorised takeoff airport matching (reuses ap_coords from cell 30) ────────
to_idx <- which(ac_data$takeoff)
if (length(to_idx) > 0) {
  to_coords       <- cbind(ac_data$lon[to_idx], ac_data$lat[to_idx])
  # ── Chunked distm to avoid OOM on large datasets ─────────────────────────
  CHUNK_SIZE      <- 5000L
  n_to            <- length(to_idx)
  nearest_to_idx  <- integer(n_to)
  nearest_to_dist <- numeric(n_to)
  for (chunk_start in seq(1L, n_to, by = CHUNK_SIZE)) {
    chunk_end  <- min(chunk_start + CHUNK_SIZE - 1L, n_to)
    chunk_rows <- chunk_start:chunk_end
    d <- distm(to_coords[chunk_rows, , drop = FALSE], ap_coords)
    nearest_to_idx[chunk_rows]  <- max.col(-d)
    nearest_to_dist[chunk_rows] <- d[cbind(seq_len(nrow(d)), nearest_to_idx[chunk_rows])]
    rm(d)
  }
  # ─────────────────────────────────────────────────────────────────────────
  ac_data$takeoff_airport_distance_m <- NA_real_
  ac_data$takeoff_airport_distance_m[to_idx] <- nearest_to_dist
  within_10km_to  <- nearest_to_dist <= 10000
  ac_data$takeoff_event_confidence[to_idx[within_10km_to]] <- 1.0


  message(sprintf("Takeoffs within 10 km of a known airport: %.1f %%",
                  100 * mean(within_10km_to)))

  src_vec <- character(nrow(ac_data))
  src_vec[to_idx[within_10km_to]] <- airports$airport[nearest_to_idx[within_10km_to]]
  ac_data[, takeoff_airport := src_vec]
  rm(src_vec)
} else {
  message("No takeoff events near a known airport")
  ac_data[, takeoff_airport := character(nrow(ac_data))]
}

# ── Propagate dest_airport and src_airport to ALL rows in each session ─────────
# dest_airport: airport where the aircraft touched down in this session
# src_airport:  airport from which the aircraft took off in this session
# rep(.N) fills every row of the session group with the same value.
ac_data[, dest_airport := {
  codes <- airport[touchdown & nchar(airport) > 0]
  rep(if (length(codes) > 0L) codes[1L] else "", .N)
}, by = id]

ac_data[, src_airport := {
  codes <- takeoff_airport[takeoff & nchar(takeoff_airport) > 0]
  rep(if (length(codes) > 0L) codes[1L] else "", .N)
}, by = id]

message("Sessions with dest_airport:         ",
        format(uniqueN(ac_data$id[nchar(ac_data$dest_airport) > 0]), big.mark = ","))
message("Sessions with src_airport:          ",
        format(uniqueN(ac_data$id[nchar(ac_data$src_airport)  > 0]), big.mark = ","))


# Aircraft Session ID Assignment

The preprocessor splits each aircraft's observations at gaps over 18 minutes and assigns S1–S4 from in-sector takeoff/landing event counts. These labels distinguish the four complete-session categories used by `03-Visualization`; the exporter cross-checks the event signature, valid track points, collection clipping, and parked/re-entry cases.

| # | Scenario | Session start | Session end |
|---|----------|---------------|-------------|
| 1 | Takeoff **and** landing inside sector (S1) | Takeoff time | Landing time |
| 2 | **Landing only** (S2 arrival) | Inbound track start | Landing time |
| 3 | **Takeoff only** (S3 departure) | Takeoff time | Non-clipped track end |
| 4 | Neither event detected (S4 transit candidate) | Track start | Track end |

The configured edge bands are not reached by this collection. Therefore S2–S4 start/exit interpretations use the upstream session label and track evidence when a direct edge crossing is unavailable; the exporter records edge evidence when present and rejects collection-clipped sessions. S4 additionally excludes parked/ground-only tracks.

**Post-processing rules applied in order:**

* **R1 – 18-min merge:** If the gap between the end of one raw session and the start of the next is < 18 min, the two are merged into a single session (handles missed-approach residuals).  
* **R2 – 10-min buffer:** Extend each session 10 min *before* the first takeoff and 10 min *after* the last landing; label those rows with the same session ID (captures taxi / roll-out time).

Output column: **`aircraft_session_id`** — `"<hex>_S<N>"` where N is 1-based per aircraft.

In [ ]:
# ==============================================================================
# aircraft_session_id — comprehensive session labeling
# ==============================================================================
#
# DATA-BLOCK DETECTION
#   Each aircraft block is split when consecutive observations are more than
#   SESSION_MIN_GAP (18 min) apart. This is a surveillance-gap split, not a
#   geographic sector-entry detector. S1-S4 are event-signature labels; the
#   visualization exporter validates those signatures and checks track and
#   collection coverage, using direct edge evidence when the dataset provides it.
#
# EVENT FILTERING: MISSED APPROACHES
#   Before the state machine runs, any landing that is followed by an in-sector
#   takeoff within MISSED_APPROACH_GAP (2 min) is discarded.  The upstream
#   touchdown filter (cell 28, 10-min consecutive-touchdown removal) removes
#   most residuals; this covers isolated ground-flag glitches where a single
#   touchdown row exists but the aircraft never actually stopped.
#
# 4-SCENARIO STATE MACHINE  (per time block, chronological event stream)
#   S1 — TO + LD both in sector  →  session = [TO,    LD   ]
#   S2 — LD only (no in-sector takeoff) → session = [block start, LD ]
#   S3 — TO only (no in-sector landing) → session = [TO, block end ]
#   S4 — neither event detected         → session = [block start, end]
#   Multiple TO/LD pairs within one time block produce multiple raw sessions.
#
# R1 — 18-MIN MERGE
#   Consecutive raw sessions whose gap < SESSION_MIN_GAP are merged into one.
#   Handles touch-and-go circuits, missed-approach residuals, rapid turnarounds.
#   Not incrementing i after a merge lets chains collapse in a single pass.
#
# R2 — 10-MIN BUFFER
#   After merging: extend session start 10 min before the first TO;
#   extend session end 10 min after the last LD.  Clamped to the aircraft's
#   data range.  Adjacent buffered sessions that would overlap are split at
#   their midpoint so each row carries exactly one session ID.
#
# OUTPUT COLUMNS
#   aircraft_session_id   — "<hex>_S<N>"  (1-indexed per aircraft)
#   aircraft_session_type — scenario (S1 / S2 / S3 / S4)
#   n_to_session          — takeoff count within the merged (unbuffered) session
#   n_ld_session          — landing count within the merged (unbuffered) session
#   session_duration_min  — unbuffered session length in minutes
#   session_dest_airport  — landing airport, propagated to every row in session
#   session_src_airport   — takeoff airport, propagated to every row in session
#
# UNIT / TYPE NOTES
#   ts is epoch MILLISECONDS — all thresholds are multiplied by 1 000.
#   as.numeric(dt$ts) is used throughout to prevent integer64 bit-pattern
#   corruption when bit64 is loaded (c(numeric,integer64) strips the class and
#   reinterprets raw bits, making every session boundary comparison return FALSE).
# ==============================================================================

SESSION_MIN_GAP     <- 1080 * 1000    # 18 min in milliseconds
SESSION_BUFFER      <-  600 * 1000    # 10 min in milliseconds
MISSED_APPROACH_GAP <-    2 * 60 * 1000   # 2 min in milliseconds

# ── Sector membership for TO / LD events ──────────────────────────────────────
within_sector_bounds <-
  data.table::between(ac_data$lat, lower_lat, upper_lat) &
  data.table::between(ac_data$lon, lower_lon, upper_lon)

wsb_safe <- !is.na(within_sector_bounds) & within_sector_bounds
ac_data[, takeoff_in_sector :=
  takeoff & takeoff_event_confidence >= 0.75 & wsb_safe]
ac_data[, touchdown_in_sector :=
  touchdown & touchdown_event_confidence >= 0.75 & wsb_safe]

rm(wsb_safe)

# ── Per-aircraft session builder ───────────────────────────────────────────────
.assign_aircraft_sessions <- function(aircraft_id, ts, takeoff_in_sector, touchdown_in_sector, min_gap, buf, ma_gap) {
  n <- length(ts)
  if (n == 0L) {
    return(list(
      aircraft_session_id   = character(0),
      aircraft_session_type = character(0),
      n_to_session          = integer(0),
      n_ld_session          = integer(0),
      session_duration_min  = numeric(0)
    ))
  }

  ts_num <- as.numeric(ts)

  # Visit boundaries from raw ts gaps (correct ms units)
  ts_gaps <- c(min_gap + 1, diff(ts_num))
  vstarts  <- sort(unique(c(1L, which(ts_gaps > min_gap))))
  vends    <- c(vstarts[-1L] - 1L, n)

  # Raw in-sector event timestamps
  to_rows_all   <- which(takeoff_in_sector   == TRUE)
  ld_rows_all   <- which(touchdown_in_sector == TRUE)
  all_to_ts_raw <- if (length(to_rows_all) > 0L) ts_num[to_rows_all] else double(0)
  all_ld_ts_raw <- if (length(ld_rows_all) > 0L) ts_num[ld_rows_all] else double(0)

  # ── Missed-approach filter ────────────────────────────────────────────────
  # Remove any landing event whose next in-sector takeoff follows within ma_gap.
  # The upstream 10-min consecutive-TD filter handles most cases; this covers
  # isolated ground-flag glitches at airports where a single TD row exists but
  # the aircraft never truly stopped.
  if (length(all_ld_ts_raw) > 0L && length(all_to_ts_raw) > 0L) {
    keep_ld <- vapply(all_ld_ts_raw, function(ld_t) {
      later <- all_to_ts_raw[all_to_ts_raw > ld_t]
      length(later) == 0L || (later[1L] - ld_t) >= ma_gap
    }, logical(1L))
    all_ld_ts <- all_ld_ts_raw[keep_ld]
  } else {
    all_ld_ts <- all_ld_ts_raw
  }
  all_to_ts <- all_to_ts_raw

  # Session accumulators (plain double — no integer64)
  ss_v     <- double(0);  se_v     <- double(0)
  has_to_v <- logical(0); has_ld_v <- logical(0)

  # ── 4-scenario state machine, one visit at a time ─────────────────────────
  for (vi in seq_along(vstarts)) {
    v_s <- ts_num[ vstarts[vi] ]
    v_e <- ts_num[ vends[vi]   ]
    if (is.na(v_s) || is.na(v_e)) next

    to_v <- sort(all_to_ts[ all_to_ts >= v_s & all_to_ts <= v_e ])
    ld_v <- sort(all_ld_ts[ all_ld_ts >= v_s & all_ld_ts <= v_e ])

    if (length(to_v) == 0L && length(ld_v) == 0L) {
      # Scenario 4: no events detected → whole time block is one session
      ss_v     <- c(ss_v,     v_s);   se_v     <- c(se_v,     v_e)
      has_to_v <- c(has_to_v, FALSE); has_ld_v <- c(has_ld_v, FALSE)
      next
    }

    evts <- data.frame(
      ts    = c(to_v, ld_v),
      etype = c(rep(1L, length(to_v)), rep(2L, length(ld_v))),
      stringsAsFactors = FALSE
    )
    evts <- evts[ order(evts$ts), , drop = FALSE ]

    cur_ss <- v_s; cur_to <- FALSE; airborne <- FALSE

    for (j in seq_len(nrow(evts))) {
      ev_ts <- evts$ts[j]
      ev_et <- evts$etype[j]

      if (ev_et == 1L && !airborne) {
        # Takeoff — session starts here (Scenarios 1 or 3)
        cur_ss <- ev_ts; cur_to <- TRUE; airborne <- TRUE

      } else if (ev_et == 2L) {
        # Landing — close current session:
        #   airborne == TRUE  → Scenario 1 (paired with an in-sector TO)
        #   airborne == FALSE → Scenario 2 (arrived airborne, no in-sector TO)
        ss_v     <- c(ss_v,     cur_ss); se_v     <- c(se_v,     ev_ts)
        has_to_v <- c(has_to_v, cur_to); has_ld_v <- c(has_ld_v, TRUE)
        cur_ss <- ev_ts; cur_to <- FALSE; airborne <- FALSE
      }
      # Spurious double-TO (already airborne): ignored
    }

    if (airborne) {
      # Scenario 3: took off but never landed within this visit
      ss_v     <- c(ss_v,     cur_ss); se_v     <- c(se_v,     v_e)
      has_to_v <- c(has_to_v, TRUE);   has_ld_v <- c(has_ld_v, FALSE)
    }
  }  # end visit loop

  if (length(ss_v) == 0L) {
    return(list(
      aircraft_session_id   = rep(NA_character_, n),
      aircraft_session_type = rep(NA_character_, n),
      n_to_session          = rep(NA_integer_, n),
      n_ld_session          = rep(NA_integer_, n),
      session_duration_min  = rep(NA_real_, n)
    ))
  }

  sdf <- data.frame(ss = ss_v, se = se_v,
                    has_to = has_to_v, has_ld = has_ld_v,
                    stringsAsFactors = FALSE)
  sdf <- sdf[ order(sdf$ss), , drop = FALSE ]
  rownames(sdf) <- NULL

  # ── R1: 18-min merge ───────────────────────────────────────────────────────
  # Not advancing i on a merge lets consecutive close-sessions chain-collapse.
  repeat {
    merged <- FALSE; i <- 1L
    while (i < nrow(sdf)) {
      gap <- sdf$ss[i + 1L] - sdf$se[i]
      if (!is.na(gap) && gap < min_gap) {
        sdf$se[i]     <- max(sdf$se[i], sdf$se[i + 1L])
        sdf$has_to[i] <- sdf$has_to[i] | sdf$has_to[i + 1L]
        sdf$has_ld[i] <- sdf$has_ld[i] | sdf$has_ld[i + 1L]
        sdf            <- sdf[-(i + 1L), , drop = FALSE]; rownames(sdf) <- NULL
        merged <- TRUE
      } else { i <- i + 1L }
    }
    if (!merged) break
  }

  # ── Scenario type, event counts, duration (all from unbuffered boundaries) ─
  sdf$scenario <- ifelse(sdf$has_to  & sdf$has_ld,  "S1",
                  ifelse(!sdf$has_to & sdf$has_ld,   "S2",
                  ifelse(sdf$has_to  & !sdf$has_ld,  "S3", "S4")))

  count_events <- function(ev_ts, ss, se)
    as.integer(sum(!is.na(ev_ts) & ev_ts >= ss & ev_ts <= se))

  sdf$n_to     <- vapply(seq_len(nrow(sdf)),
    function(i) count_events(all_to_ts, sdf$ss[i], sdf$se[i]), integer(1L))
  sdf$n_ld     <- vapply(seq_len(nrow(sdf)),
    function(i) count_events(all_ld_ts, sdf$ss[i], sdf$se[i]), integer(1L))
  sdf$dur_min  <- round((sdf$se - sdf$ss) / 60000, 1L)   # ms → minutes

  # ── R2: 10-min buffer ──────────────────────────────────────────────────────
  sdf$ss_buf <- sdf$ss; sdf$se_buf <- sdf$se

  for (i in seq_len(nrow(sdf))) {
    if (sdf$has_to[i]) {
      to_in <- all_to_ts[ all_to_ts >= sdf$ss[i] & all_to_ts <= sdf$se[i] ]
      if (length(to_in) > 0L) sdf$ss_buf[i] <- min(to_in) - buf
    }
    if (sdf$has_ld[i]) {
      ld_in <- all_ld_ts[ all_ld_ts >= sdf$ss[i] & all_ld_ts <= sdf$se[i] ]
      if (length(ld_in) > 0L) sdf$se_buf[i] <- max(ld_in) + buf
    }
  }

  # Clamp to aircraft data range
  sdf$ss_buf <- pmax(sdf$ss_buf, ts_num[1L])
  sdf$se_buf <- pmin(sdf$se_buf, ts_num[n])

  # Resolve any buffer overlaps: split at midpoint (no row double-labeled)
  for (i in seq_len(nrow(sdf) - 1L)) {
    if (!is.na(sdf$se_buf[i]) && !is.na(sdf$ss_buf[i + 1L]) &&
        sdf$se_buf[i] >= sdf$ss_buf[i + 1L]) {
      mid <- (sdf$se_buf[i] + sdf$ss_buf[i + 1L]) / 2
      sdf$se_buf[i] <- mid; sdf$ss_buf[i + 1L] <- mid + 1
    }
  }

  # ── Label rows ─────────────────────────────────────────────────────────────
  session_ids  <- rep(NA_character_, n)
  session_type <- rep(NA_character_, n)
  n_to_vec     <- rep(NA_integer_,   n)
  n_ld_vec     <- rep(NA_integer_,   n)
  dur_vec      <- rep(NA_real_,      n)

  for (s in seq_len(nrow(sdf))) {
    if (is.na(sdf$ss_buf[s]) || is.na(sdf$se_buf[s])) next
    rows <- ts_num >= sdf$ss_buf[s] & ts_num <= sdf$se_buf[s]
    rows[is.na(rows)] <- FALSE
    if (!any(rows)) next
    session_ids[rows]  <- paste0(aircraft_id, "_S", s)
    session_type[rows] <- sdf$scenario[s]
    n_to_vec[rows]     <- sdf$n_to[s]
    n_ld_vec[rows]     <- sdf$n_ld[s]
    dur_vec[rows]      <- sdf$dur_min[s]
  }

  return(list(
    aircraft_session_id   = session_ids,
    aircraft_session_type = session_type,
    n_to_session          = n_to_vec,
    n_ld_session          = n_ld_vec,
    session_duration_min  = dur_vec
  ))
}

# ── Apply to all aircraft ──────────────────────────────────────────────────────
message("Building aircraft_session_id for ",
        format(uniqueN(ac_data$aircraft), big.mark = ","), " aircraft ...")

setorder(ac_data, aircraft, ts)
ac_data[, c("aircraft_session_id", "aircraft_session_type", "n_to_session", "n_ld_session", "session_duration_min") :=
  .assign_aircraft_sessions(
    aircraft_id         = aircraft[1L],
    ts                  = ts,
    takeoff_in_sector   = takeoff_in_sector,
    touchdown_in_sector = touchdown_in_sector,
    min_gap             = SESSION_MIN_GAP,
    buf                 = SESSION_BUFFER,
    ma_gap              = MISSED_APPROACH_GAP
  ),
  by = aircraft
]
invisible(gc(full = TRUE))


# Hand-picking numeric variables to reduce floating precision

In [ ]:
numerics <- colnames(ac_data)[sapply(ac_data, class) == 'numeric']
numerics

##############################
########### Output ###########
##############################
#  [1] "V1"                        "lat"                      
#  [3] "lon"                       "altitude"                 
#  [5] "ground_flag"               "quadrant"                 
#  [7] "last_lat"                  "last_lon"                 
#  [9] "last_altitude"             "last_ground_flag"         
# [11] "last_quadrant"             "next_lat"                 
# [13] "next_lon"                  "next_altitude"            
# [15] "next_ground_flag"          "next_quadrant"            
# [17] "max_ac_alt"                "est_climb_rate"           
# [19] "climb_rate_ratio"          "max_altitude_touchdown_id"
# [21] "touchdown_time_diff"

Unfortunately float::fl doesn't work as expected as it just stores an 4 byte integer that's ready for converesion back to 8 byte float

In [ ]:
# ==============================================================================
# Type compression (reduce memory footprint)
# ==============================================================================
ac_data$altitude          <- as.integer(ac_data$altitude)
ac_data$ground_flag       <- (ac_data$ground_flag == 1)
ac_data$quadrant          <- as.integer(ac_data$quadrant)
ac_data$last_altitude     <- as.integer(ac_data$last_altitude)
ac_data$last_ground_flag  <- (ac_data$last_ground_flag == 1)
ac_data$last_quadrant     <- as.integer(ac_data$last_quadrant)
ac_data$next_altitude     <- as.integer(ac_data$next_altitude)
ac_data$next_ground_flag  <- (ac_data$next_ground_flag == 1)
ac_data$next_quadrant     <- as.integer(ac_data$next_quadrant)
ac_data$max_ac_alt        <- as.integer(ac_data$max_ac_alt)
ac_data$max_alt_touchdown_seg <- as.integer(ac_data$max_alt_touchdown_seg)

# Altitude ratio filter

In [ ]:
# ==============================================================================
# Altitude ratio filter
# ==============================================================================
# Keep all observations. Flag the legacy ceiling for review instead of
# removing individual samples or entire flight IDs from the canonical output.
ac_data$altitude_ceiling_flag <- is.finite(ac_data$altitude_raw_ft) &
  ac_data$altitude_raw_ft > altitude_ceiling
quality_flag_columns <- c(
  "track_gap_flag", "position_invalid_flag", "altitude_invalid_flag",
  "altitude_range_flag", "position_jump_suspect_flag",
  "altitude_jump_suspect_flag", "altitude_spike_suspect_flag",
  "altitude_ceiling_flag"
)
ac_data[, quality_flag_count := rowSums(.SD), .SDcols = quality_flag_columns]
message("Rows above configured altitude ceiling: ",
        format(sum(ac_data$altitude_ceiling_flag, na.rm=TRUE), big.mark=","))
message("Rows retained after non-destructive QC: ", format(nrow(ac_data), big.mark=","))


# Saving final prepropcessed data

In [ ]:
# ==============================================================================
# Save outputs
# ==============================================================================
setDT(ac_data)
setorder(ac_data, aircraft, ts)

# feather 0.3.5 mis-serializes bit64::integer64 as a subnormal double.
# Epoch milliseconds are exact as doubles at this magnitude (< 2^53).
ac_data[, ts := as.numeric(ts)]

# Core processed dataset (minimal columns)
write_feather(
  ac_data[, .(aircraft, id, aircraft_session_id,
              aircraft_session_type,
              n_to_session, n_ld_session, session_duration_min,
              ts, ts_readable, lon, lat, altitude_raw_ft, altitude_phase_ft,
              altitude, ground_speed, azimuth, ground_flag,
              ground_flag_confidence, track_segment_id, sample_gap_ms,
              position_speed_knots, altitude_rate_fpm,
              track_gap_flag, position_invalid_flag, altitude_invalid_flag,
              altitude_range_flag, position_jump_suspect_flag,
              altitude_jump_suspect_flag, altitude_spike_suspect_flag,
              altitude_ceiling_flag, quality_flag_count,
              takeoff_event_confidence, touchdown_event_confidence,
              src_airport, dest_airport,
              session_src_airport, session_dest_airport)],

  "ac_data_processed.feather"
)

# Construct forward/backward markers for touchdown and entry events
ac_data[,
  c("last_touchdown", "next_touchdown",
    "last_takeoff",   "next_takeoff",
    "last_entered",   "next_entered",
    "last_airport",   "next_airport") :=
  list(
    c(FALSE, touchdown[-.N]), c(touchdown[-1], FALSE),
    c(FALSE, takeoff[-.N]),   c(takeoff[-1],   FALSE),
    c(FALSE, entered[-.N]),   c(entered[-1],   FALSE),
    c("",    airport[-.N]),   c(airport[-1],   "")
  ),
  by = "aircraft"
]
ac_data$next_touchdown[is.na(ac_data$next_touchdown)] <- FALSE
ac_data$next_takeoff[is.na(ac_data$next_takeoff)]     <- FALSE

In [ ]:
write_feather(ac_data, "full_ac_data.feather")
colnames(ac_data)

In [ ]:
c(length(unique(ac_data$aircraft)), length(unique(ac_data$aircraft_session_id)), nrow(ac_data))

In [ ]:
jfk_data <- ac_data[ac_data$dest_airport == "JFK", ]
# jfk_data <- data.frame(id = unique(ac_data$id[ac_data$next_airport == "JFK" & ac_data$next_touchdown]), jfk_landing_flag = T)
# jfk_data <- merge(jfk_data, ac_data, all.x = T, all.y = T)
# jfk_data$jfk_landing_flag[is.na(jfk_data$jfk_landing_flag)] <- F
setDT(jfk_data)
jfk_data[, "max_id_alt" := max(altitude), by = "id"]
write.csv(jfk_data[, c("aircraft_session_id", "aircraft", "lat", "lon", "ts", "altitude",# "climb_rate",
                       "azimuth", "ground_speed",# "jfk_landing_flag",
                       "ground_flag")],
          "samples_with_jfk_landing_flag.csv", row.names = F)
# saveRDS(jfk_data[jfk_data$max_id_alt >= 20000, ], "high_ac_data.Rds")
write_feather(jfk_data, "full_jfk_data.feather")